# **Start Section:**


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip install git+https://github.com/DaneshSelwal/treeffuser.git


In [ ]:
!pip install --quiet numpy==1.26.4 pandas scipy scikit-learn matplotlib seaborn xlsxwriter openpyxl torch properscoring


In [ ]:
import os
os.environ["PIP_CONSTRAINT"] = "/tmp/numpy_constraint.txt"
!echo "numpy==1.26.4" > /tmp/numpy_constraint.txt


# **Imports**


In [ ]:
import io
import os
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import properscoring as ps
import torch
import torch.nn as nn
from scipy.stats import kstest, norm, pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [ ]:
# Go to find & replace button and replace (Data_folder) with your folder name. Rename your train and test dataset as train.csv and test.csv.
# Modify the names of the feature in the next cell if your input features are different.
target_column = None  # Keep None to automatically use the last column as the target.
random_seed = 42


In [ ]:
feature_names = ['Qt', 'Qt-1', 'St-1']


In [ ]:
train_data_path = "./drive/MyDrive/Data_folder/Data/train.csv"
test_data_path = "./drive/MyDrive/Data_folder/Data/test.csv"
output_folder = "./drive/MyDrive/Data_folder/Hyperspherical_Confidence_Mapping(HCM)"


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_demo_dataframe(n_rows, feature_names, seed=42, is_train=True):
    rng = np.random.default_rng(seed + (0 if is_train else 99))
    data = {name: rng.normal(loc=0.0, scale=1.0, size=n_rows) for name in feature_names}
    frame = pd.DataFrame(data)
    nonlinear_term = 0.8 * np.sin(frame[feature_names[0]].values)
    interaction_term = 0.5 * frame[feature_names[1]].values * frame[feature_names[2]].values
    trend_term = 0.3 * frame[feature_names[0]].values ** 2
    noise = rng.normal(loc=0.0, scale=0.35 if is_train else 0.4, size=n_rows)
    frame['Target'] = 12.0 + 3.2 * frame[feature_names[0]].values - 1.7 * frame[feature_names[1]].values + 2.1 * frame[feature_names[2]].values + nonlinear_term + interaction_term - trend_term + noise
    return frame


def read_csv_from_candidates(path_candidates):
    for path in path_candidates:
        path = Path(path)
        if not path.exists():
            continue
        try:
            df = pd.read_csv(path)
            if not df.empty:
                print(f"Loaded data from: {path}")
                return df
        except Exception:
            continue
    return pd.DataFrame()


set_seed(random_seed)
train_demo = build_demo_dataframe(240, feature_names, seed=random_seed, is_train=True)
test_demo = build_demo_dataframe(80, feature_names, seed=random_seed, is_train=False)

train_candidates = [
    train_data_path,
    "/content/drive/MyDrive/Data_folder/Data/train.csv",
    "Data_folder/Data/train.csv"
]
test_candidates = [
    test_data_path,
    "/content/drive/MyDrive/Data_folder/Data/test.csv",
    "Data_folder/Data/test.csv"
]

train_data = read_csv_from_candidates(train_candidates)
test_data = read_csv_from_candidates(test_candidates)

if train_data.empty:
    print("Warning: training CSV is missing or empty. A generated demo dataset will be used so the notebook can still run in Google Colab.")
    train_data = train_demo.copy()
if test_data.empty:
    print("Warning: testing CSV is missing or empty. A generated demo dataset will be used so the notebook can still run in Google Colab.")
    test_data = test_demo.copy()


In [ ]:
print("\nShape of training data:", train_data.shape)
print("First 5 rows of training data:\n", train_data.head(5))
print("\nShape of test data:", test_data.shape)
print("First 5 rows of test data:\n", test_data.head(5))


In [ ]:
if target_column is None:
    target_column = train_data.columns[-1]

available_feature_names = [name for name in feature_names if name in train_data.columns]
if len(available_feature_names) == len(feature_names):
    selected_feature_names = feature_names
else:
    selected_feature_names = train_data.columns[:-1].tolist()
    print(f"Feature names were adjusted automatically to match the dataset columns: {selected_feature_names}")

X_train_full = train_data[selected_feature_names].copy()
y_train_full = train_data[target_column].copy()
X_test = test_data[selected_feature_names].copy()
y_test = test_data[target_column].copy()


In [ ]:
# Apply z-score normalization
X_train_raw, X_val_raw, y_train_raw, y_val_raw = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.20,
    random_state=random_seed
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val = scaler.transform(X_val_raw)
X_test_scaled = scaler.transform(X_test)

y_train = y_train_raw.to_numpy(dtype=np.float32).reshape(-1, 1)
y_val = y_val_raw.to_numpy(dtype=np.float32).reshape(-1, 1)
y_test_array = y_test.to_numpy(dtype=np.float32).reshape(-1, 1)

print("\nSelected feature names:", selected_feature_names)
print("Target column:", target_column)
print("Training split:", X_train.shape, y_train.shape)
print("Validation split:", X_val.shape, y_val.shape)
print("Test split:", X_test_scaled.shape, y_test_array.shape)


# **Functions:**


In [ ]:
def expand_scalar_targets(y_array):
    y_array = np.asarray(y_array, dtype=np.float32).reshape(-1, 1)
    return np.concatenate([y_array, y_array], axis=1)


def create_dataloader(X_array, y_array, batch_size=32, shuffle=True):
    features = torch.tensor(X_array, dtype=torch.float32)
    expanded_targets = torch.tensor(expand_scalar_targets(y_array), dtype=torch.float32)
    dataset = torch.utils.data.TensorDataset(features, expanded_targets)
    return torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


class HCMRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dims=(128, 64, 32), dropout=0.10):
        super().__init__()
        layers = []
        last_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(last_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            last_dim = hidden_dim
        self.backbone = nn.Sequential(*layers)
        self.direction_head = nn.Linear(last_dim, 2)
        self.magnitude_head = nn.Sequential(
            nn.Linear(last_dim, 1),
            nn.Softplus()
        )

    def forward(self, x):
        hidden = self.backbone(x)
        pred_direction = self.direction_head(hidden)
        pred_magnitude = self.magnitude_head(hidden)
        return pred_magnitude, pred_direction


def compute_hcm_targets(expanded_targets):
    target_magnitude = torch.linalg.norm(expanded_targets, dim=1, keepdim=True)
    target_direction = expanded_targets / (target_magnitude + 1e-6)
    return target_magnitude, target_direction


def hcm_loss(pred_magnitude, pred_direction, expanded_targets, lambda_norm=0.20):
    target_magnitude, target_direction = compute_hcm_targets(expanded_targets)
    magnitude_loss = nn.functional.smooth_l1_loss(pred_magnitude, target_magnitude)
    direction_loss = nn.functional.mse_loss(pred_direction * target_magnitude, expanded_targets)
    norm_penalty = nn.functional.smooth_l1_loss(
        torch.linalg.norm(pred_direction, dim=1, keepdim=True),
        torch.ones_like(pred_magnitude)
    )
    total_loss = magnitude_loss + direction_loss + lambda_norm * norm_penalty
    return total_loss, {
        'total_loss': float(total_loss.item()),
        'magnitude_loss': float(magnitude_loss.item()),
        'direction_loss': float(direction_loss.item()),
        'norm_penalty': float(norm_penalty.item())
    }


def predict_with_hcm(model, X_array, device):
    model.eval()
    with torch.no_grad():
        X_tensor = torch.tensor(X_array, dtype=torch.float32, device=device)
        pred_magnitude, pred_direction = model(X_tensor)
        pred_norm = torch.linalg.norm(pred_direction, dim=1, keepdim=True)
        pred_expanded = pred_magnitude * pred_direction
        pred_mean = pred_expanded[:, :1]
        raw_uncertainty = pred_magnitude * torch.abs(pred_norm - 1.0)
    return {
        'pred_mean': pred_mean.cpu().numpy().reshape(-1),
        'pred_magnitude': pred_magnitude.cpu().numpy().reshape(-1),
        'pred_direction': pred_direction.cpu().numpy(),
        'pred_norm': pred_norm.cpu().numpy().reshape(-1),
        'raw_uncertainty': raw_uncertainty.cpu().numpy().reshape(-1)
    }


def calibrate_uncertainty_scale(raw_uncertainty, absolute_errors):
    raw_uncertainty = np.asarray(raw_uncertainty, dtype=float)
    absolute_errors = np.asarray(absolute_errors, dtype=float)
    raw_uncertainty = np.clip(raw_uncertainty, 1e-6, None)
    ratio = absolute_errors / raw_uncertainty
    scale = np.quantile(ratio, 0.90)
    return max(float(scale), 1e-3)


def train_hcm_model(X_train, y_train, X_val, y_val, training_config):
    set_seed(training_config['seed'])
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    train_loader = create_dataloader(X_train, y_train, batch_size=training_config['batch_size'], shuffle=True)

    model = HCMRegressor(
        input_dim=X_train.shape[1],
        hidden_dims=training_config['hidden_dims'],
        dropout=training_config['dropout']
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=training_config['learning_rate'],
        weight_decay=training_config['weight_decay']
    )
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=training_config['scheduler_step'],
        gamma=training_config['scheduler_gamma']
    )

    history = []
    best_state = None
    best_val_loss = np.inf
    patience_counter = 0

    for epoch in range(training_config['epochs']):
        model.train()
        batch_losses = []
        for batch_features, batch_targets in train_loader:
            batch_features = batch_features.to(device)
            batch_targets = batch_targets.to(device)

            optimizer.zero_grad()
            pred_magnitude, pred_direction = model(batch_features)
            loss, loss_details = hcm_loss(
                pred_magnitude,
                pred_direction,
                batch_targets,
                lambda_norm=training_config['lambda_norm']
            )
            loss.backward()
            optimizer.step()
            batch_losses.append(loss_details['total_loss'])

        scheduler.step()

        val_predictions = predict_with_hcm(model, X_val, device)
        val_loss = np.mean((val_predictions['pred_mean'] - y_val.reshape(-1)) ** 2)
        history.append({'epoch': epoch + 1, 'train_loss': float(np.mean(batch_losses)), 'val_loss': float(val_loss)})

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch + 1) % training_config['print_every'] == 0 or epoch == 0:
            print(f"Epoch {epoch + 1}/{training_config['epochs']} | Train Loss: {np.mean(batch_losses):.5f} | Val Loss: {val_loss:.5f}")

        if patience_counter >= training_config['patience']:
            print(f"Early stopping triggered at epoch {epoch + 1}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    val_predictions = predict_with_hcm(model, X_val, device)
    val_abs_error = np.abs(val_predictions['pred_mean'] - y_val.reshape(-1))
    uncertainty_scale = calibrate_uncertainty_scale(val_predictions['raw_uncertainty'], val_abs_error)
    print(f"Calibrated uncertainty scale: {uncertainty_scale:.5f}")

    return model, history, uncertainty_scale, device


In [ ]:
def create_predictions_dataframe(prediction_dict, y_true, uncertainty_scale):
    y_true = np.asarray(y_true).reshape(-1)
    raw_uncertainty = np.clip(prediction_dict['raw_uncertainty'], 1e-6, None)
    std_proxy = np.clip(raw_uncertainty * uncertainty_scale, 1e-6, None)
    mean_prediction = prediction_dict['pred_mean']
    lower_bound = mean_prediction - 1.96 * std_proxy
    upper_bound = mean_prediction + 1.96 * std_proxy
    absolute_error = np.abs(y_true - mean_prediction)
    coverage_95 = ((y_true >= lower_bound) & (y_true <= upper_bound)).astype(int)

    return pd.DataFrame({
        'Mean': mean_prediction,
        'StdDev': std_proxy,
        'Lower_95': lower_bound,
        'Upper_95': upper_bound,
        'RawUncertainty': raw_uncertainty,
        'DirectionNorm': prediction_dict['pred_norm'],
        'Magnitude': prediction_dict['pred_magnitude'],
        'Absolute_Error': absolute_error,
        'Coverage_95': coverage_95,
        'Actual': y_true
    })


def insert_figure_into_worksheet(writer, sheet_name, figure, image_cell='H2'):
    image_buffer = io.BytesIO()
    figure.savefig(image_buffer, format='png', dpi=200, bbox_inches='tight')
    image_buffer.seek(0)
    worksheet = writer.sheets[sheet_name]
    worksheet.insert_image(image_cell, 'plot.png', {'image_data': image_buffer})
    plt.close(figure)


def generate_and_save_plots(predictions_df, y_test, excel_path):
    with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
        predictions_df.to_excel(writer, sheet_name='Predictions', index=False)

        plt.figure(figsize=(12, 8))
        sns.scatterplot(x='Actual', y='Mean', data=predictions_df)
        plt.plot([predictions_df['Actual'].min(), predictions_df['Actual'].max()],
                 [predictions_df['Actual'].min(), predictions_df['Actual'].max()],
                 color='red', linestyle='--')
        plt.title('Predicted Y_Label vs. Actual Y_Label')
        plt.xlabel('Actual Y_Label')
        plt.ylabel('Predicted Mean Y_Label')
        plt.tight_layout()
        predicted_vs_actual_path = 'predicted_vs_actual.png'
        plt.savefig(predicted_vs_actual_path)
        plt.close()

        plt.figure(figsize=(12, 8))
        plt.errorbar(x=range(len(predictions_df)), y=predictions_df['Mean'],
                     yerr=predictions_df['StdDev'], fmt='o', markersize=4, ecolor='red', elinewidth=1, capsize=5, alpha=0.7)
        plt.plot(range(len(predictions_df)), predictions_df['Mean'], color='black', linewidth=0.5, alpha=0.7)
        plt.title('Predicted Mean with Uncertainty Bands', fontsize=16)
        plt.xlabel('Sample Number', fontsize=14)
        plt.ylabel('Predicted Mean Y_Label')
        plt.tight_layout()
        uncertainty_bands_path = 'uncertainty_bands.png'
        plt.savefig(uncertainty_bands_path)
        plt.close()

        plt.figure(figsize=(12, 8))
        sns.histplot(predictions_df['StdDev'], bins=20, kde=True)
        plt.title('Distribution of Predicted Standard Deviations')
        plt.xlabel('Standard Deviation')
        plt.ylabel('Frequency')
        plt.tight_layout()
        histogram_stddev_path = 'histogram_stddev.png'
        plt.savefig(histogram_stddev_path)
        plt.close()

        residuals = predictions_df['Actual'] - predictions_df['Mean']
        plt.figure(figsize=(12, 8))
        sns.scatterplot(x=predictions_df['Mean'], y=residuals)
        plt.axhline(0, color='red', linestyle='--')
        plt.title('Residual Plot')
        plt.xlabel('Predicted Mean Y_Label')
        plt.ylabel('Residuals')
        plt.tight_layout()
        residual_plot_path = 'residual_plot.png'
        plt.savefig(residual_plot_path)
        plt.close()

        worksheet = writer.book.add_worksheet('Plots')
        writer.sheets['Plots'] = worksheet
        worksheet.insert_image('B2', predicted_vs_actual_path)
        worksheet.insert_image('B25', uncertainty_bands_path)
        worksheet.insert_image('N2', histogram_stddev_path)
        worksheet.insert_image('N25', residual_plot_path)


In [ ]:
def generate_and_save_crps_decomposition(
    predictions_df,
    y_true,
    excel_path,
    model_name="Model",
    n_bins=10
):
    mu_pred = predictions_df['Mean'].values
    sigma_pred = np.clip(predictions_df['StdDev'].values, 1e-6, None)

    if 'Actual' in predictions_df.columns:
        y_true = predictions_df['Actual'].values
    else:
        y_true = np.asarray(y_true).ravel()

    y_true = np.asarray(y_true).ravel()

    crps_values = ps.crps_gaussian(y_true, mu=mu_pred, sig=sigma_pred)
    total_crps = np.mean(crps_values)

    global_mu = np.mean(y_true)
    global_sigma = np.std(y_true) + 1e-6
    uncertainty = np.mean(ps.crps_gaussian(y_true, mu=global_mu, sig=global_sigma))

    sort_idx = np.argsort(mu_pred)
    y_sorted = y_true[sort_idx]
    mu_sorted = mu_pred[sort_idx]
    sigma_sorted = sigma_pred[sort_idx]

    bins = np.array_split(np.arange(len(y_true)), n_bins)
    bin_results = []
    reliability_sum = 0.0

    for i, b in enumerate(bins):
        bin_y = y_sorted[b]
        bin_mu = mu_sorted[b]
        bin_sigma = sigma_sorted[b]
        bin_crps = np.mean(ps.crps_gaussian(bin_y, mu=bin_mu, sig=bin_sigma))
        reliability_sum += bin_crps
        bin_results.append({
            "Bin": i + 1,
            "Bin_Size": len(b),
            "Mean_Pred_Mu": np.mean(bin_mu),
            "Mean_Pred_Sigma": np.mean(bin_sigma),
            "Bin_CRPS": bin_crps
        })

    reliability = reliability_sum / n_bins
    resolution = uncertainty - (total_crps - reliability)

    summary_df = pd.DataFrame([{
        "Model": model_name,
        "Total_CRPS": total_crps,
        "Uncertainty": uncertainty,
        "Reliability": reliability,
        "Resolution": resolution
    }])
    bins_df = pd.DataFrame(bin_results)

    with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        bins_df.to_excel(writer, sheet_name='CRPS_Bins', index=False)
        summary_df.to_excel(writer, sheet_name='CRPS_Summary', index=False)


In [ ]:
def generate_and_save_pit_results(
    predictions_df,
    y_true,
    excel_path,
    model_name="Model",
    n_bins=10
):
    mu_pred = predictions_df['Mean'].values
    sigma_pred = np.clip(predictions_df['StdDev'].values, 1e-6, None)

    if 'Actual' in predictions_df.columns:
        y_true = predictions_df['Actual'].values
    else:
        y_true = np.asarray(y_true).ravel()

    y_true = np.asarray(y_true).ravel()
    pit_values = norm.cdf(y_true, loc=mu_pred, scale=sigma_pred)
    pit_df = pd.DataFrame({"PIT_Value": pit_values})

    hist_density, bin_edges = np.histogram(pit_values, bins=n_bins, range=(0, 1), density=True)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    hist_df = pd.DataFrame({"Bin_Center": bin_centers, "Density": hist_density})

    pit_mean = pit_values.mean()
    pit_var = pit_values.var()
    ks_stat, ks_pvalue = kstest(pit_values, 'uniform')

    if ks_pvalue > 0.05:
        diagnosis = "Well-calibrated (Uniform PIT)"
    elif pit_var < 1/12:
        diagnosis = "Over-dispersed (Under-confident)"
    elif pit_var > 1/12:
        diagnosis = "Under-dispersed (Over-confident)"
    else:
        diagnosis = "Biased or miscalibrated"

    summary_df = pd.DataFrame([{
        "Model": model_name,
        "Mean_PIT": pit_mean,
        "Variance_PIT": pit_var,
        "Ideal_Variance_(1/12)": 1/12,
        "KS_Statistic": ks_stat,
        "KS_pvalue": ks_pvalue,
        "Diagnosis": diagnosis
    }])

    with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        pit_df.to_excel(writer, sheet_name='PIT_Values', index=False)
        hist_df.to_excel(writer, sheet_name='PIT_Histogram', index=False)
        summary_df.to_excel(writer, sheet_name='PIT_Summary', index=False)


In [ ]:
def generate_and_save_calibration_curve(
    predictions_df,
    y_true,
    excel_path,
    model_name="Model",
    n_quantiles=10
):
    mu_pred = predictions_df['Mean'].values
    sigma_pred = np.clip(predictions_df['StdDev'].values, 1e-6, None)

    if 'Actual' in predictions_df.columns:
        y_true = predictions_df['Actual'].values
    else:
        y_true = np.asarray(y_true).ravel()

    y_true = np.asarray(y_true).ravel()

    quantiles = np.linspace(0, 1, n_quantiles + 1)
    observed_freq = []
    predicted_q = []

    for q in quantiles:
        thresh = norm.ppf(q, loc=mu_pred, scale=sigma_pred)
        obs = np.mean(y_true <= thresh)
        predicted_q.append(q)
        observed_freq.append(obs)

    predicted_q = np.array(predicted_q)
    observed_freq = np.array(observed_freq)

    mce = np.mean(np.abs(observed_freq - predicted_q))
    rmsce = np.sqrt(np.mean((observed_freq - predicted_q) ** 2))

    calib_df = pd.DataFrame({
        "Predicted_Quantile": predicted_q,
        "Observed_Frequency": observed_freq,
        "Absolute_Error": np.abs(observed_freq - predicted_q)
    })

    summary_df = pd.DataFrame([{
        "Model": model_name,
        "Mean_Calibration_Error (MCE)": mce,
        "RMS_Calibration_Error (RMSCE)": rmsce,
        "Num_Quantiles": n_quantiles
    }])

    with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        calib_df.to_excel(writer, sheet_name='Calibration_Curve', index=False)
        summary_df.to_excel(writer, sheet_name='Calibration_Summary', index=False)


In [ ]:
# Define the path to the folder
folder_path = "/content/drive/MyDrive/Data_folder"
os.makedirs(folder_path, exist_ok=True)

if 'google.colab' in sys.modules:
    output_candidates = [
        Path(output_folder),
        Path('/content/drive/MyDrive/Data_folder/Hyperspherical_Confidence_Mapping(HCM)')
    ]
else:
    output_candidates = [
        Path('Data_folder/Hyperspherical_Confidence_Mapping(HCM)'),
        Path(output_folder)
    ]

for candidate in output_candidates:
    try:
        candidate.mkdir(parents=True, exist_ok=True)
        resolved_output_folder = candidate
        break
    except Exception:
        continue
else:
    raise RuntimeError('Could not create an output folder for HCM results.')


def generate_and_save_error_alignment(predictions_df, excel_path, model_name='HCM'):
    diagnostics_df = predictions_df[['RawUncertainty', 'StdDev', 'Absolute_Error', 'Coverage_95']].copy()
    diagnostics_df['Confidence'] = np.exp(-diagnostics_df['StdDev'])
    diagnostics_df['Absolute_Error_Rank'] = diagnostics_df['Absolute_Error'].rank(method='average')
    diagnostics_df['StdDev_Rank'] = diagnostics_df['StdDev'].rank(method='average')

    pearson_value = pearsonr(diagnostics_df['StdDev'], diagnostics_df['Absolute_Error'])[0]
    spearman_value = spearmanr(diagnostics_df['StdDev'], diagnostics_df['Absolute_Error'])[0]

    summary_df = pd.DataFrame([{
        'Model': model_name,
        'Pearson_Correlation': pearson_value,
        'Spearman_Correlation': spearman_value,
        'Average_Coverage_95': diagnostics_df['Coverage_95'].mean(),
        'Average_StdDev': diagnostics_df['StdDev'].mean(),
        'Average_Absolute_Error': diagnostics_df['Absolute_Error'].mean()
    }])

    with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        diagnostics_df.to_excel(writer, sheet_name='Error_Alignment', index=False)
        summary_df.to_excel(writer, sheet_name='Alignment_Summary', index=False)


def plot_metrics_and_save_to_excel(writer, predictions_df, y_true, model_name):
    mean_pred = predictions_df['Mean']
    std_pred = np.clip(predictions_df['StdDev'], 1e-6, None)

    sharpness = std_pred.mean()
    crps_values = ps.crps_gaussian(y_true, mean_pred, std_pred)
    nll_values = -norm.logpdf(y_true, loc=mean_pred, scale=std_pred)
    nll = nll_values.mean()

    metrics_df = pd.DataFrame({
        'Mean Prediction': mean_pred,
        'Standard Deviation': std_pred,
        'CRPS': crps_values,
        'NLL': nll_values
    })

    summary_df = pd.DataFrame({
        'Metric': ['Sharpness', 'Mean CRPS', 'Mean NLL'],
        'Value': [sharpness, crps_values.mean(), nll]
    })

    metrics_sheet_name = f'{model_name} Metrics'
    metrics_df.to_excel(writer, sheet_name=metrics_sheet_name, index=False)

    summary_sheet_name = f'{model_name} Summary'
    summary_df.to_excel(writer, sheet_name=summary_sheet_name, index=False)

    fig, axs = plt.subplots(2, 2, figsize=(12, 10))
    axs[0, 0].hist(std_pred, bins=30, color='skyblue', edgecolor='black')
    axs[0, 0].set_title(f'Sharpness (StdDev) for {model_name}')
    axs[0, 0].set_xlabel('Standard Deviation')
    axs[0, 0].set_ylabel('Frequency')

    axs[0, 1].hist(crps_values, bins=30, color='lightgreen', edgecolor='black')
    axs[0, 1].set_title(f'CRPS Distribution for {model_name}')
    axs[0, 1].set_xlabel('CRPS')
    axs[0, 1].set_ylabel('Frequency')

    axs[1, 0].hist(nll_values, bins=30, color='salmon', edgecolor='black')
    axs[1, 0].set_title(f'NLL Distribution for {model_name}')
    axs[1, 0].set_xlabel('NLL')
    axs[1, 0].set_ylabel('Frequency')

    axs[1, 1].scatter(std_pred, np.abs(y_true - mean_pred), alpha=0.7)
    axs[1, 1].set_title(f'Uncertainty vs Error for {model_name}')
    axs[1, 1].set_xlabel('Standard Deviation')
    axs[1, 1].set_ylabel('Absolute Error')
    plt.tight_layout()

    worksheet = writer.book.add_worksheet(f'{model_name} Plots')
    writer.sheets[f'{model_name} Plots'] = worksheet
    insert_figure_into_worksheet(writer, f'{model_name} Plots', fig, image_cell='B2')


In [ ]:
def build_matrix_evaluation(predictions_df, model_name='HCM'):
    rmse = np.sqrt(mean_squared_error(predictions_df['Actual'], predictions_df['Mean']))
    mae = mean_absolute_error(predictions_df['Actual'], predictions_df['Mean'])
    pearson_value = pearsonr(predictions_df['StdDev'], predictions_df['Absolute_Error'])[0]
    spearman_value = spearmanr(predictions_df['StdDev'], predictions_df['Absolute_Error'])[0]
    interval_coverage = predictions_df['Coverage_95'].mean()
    average_interval_width = (predictions_df['Upper_95'] - predictions_df['Lower_95']).mean()
    sharpness = predictions_df['StdDev'].mean()
    norm_violation = np.abs(predictions_df['DirectionNorm'] - 1.0).mean()

    return pd.DataFrame([{
        'Model': model_name,
        'RMSE': rmse,
        'MAE': mae,
        'Pearson': pearson_value,
        'Spearman': spearman_value,
        'Coverage_95': interval_coverage,
        'Average_Interval_Width': average_interval_width,
        'Sharpness': sharpness,
        'Mean_Norm_Violation': norm_violation
    }])


# **Hyperspherical Confidence Mapping (HCM)**


In [ ]:
training_config = {
    'seed': random_seed,
    'epochs': 160,
    'batch_size': 32,
    'learning_rate': 1e-3,
    'weight_decay': 1e-4,
    'scheduler_step': 40,
    'scheduler_gamma': 0.6,
    'hidden_dims': (128, 64, 32),
    'dropout': 0.10,
    'lambda_norm': 0.20,
    'patience': 30,
    'print_every': 20
}
training_config


In [ ]:
model, training_history, uncertainty_scale, device = train_hcm_model(
    X_train,
    y_train,
    X_val,
    y_val,
    training_config
)

test_predictions = predict_with_hcm(model, X_test_scaled, device)
predictions_HCM_df = create_predictions_dataframe(test_predictions, y_test_array, uncertainty_scale)
training_history_df = pd.DataFrame(training_history)

print(predictions_HCM_df.head())


In [ ]:
hcm_excel_path = resolved_output_folder / 'HCM.xlsx'
generate_and_save_plots(predictions_HCM_df, y_test_array, hcm_excel_path)
print(f"Saved detailed HCM outputs to: {hcm_excel_path}")


In [ ]:
generate_and_save_crps_decomposition(
    predictions_HCM_df,
    y_test_array,
    hcm_excel_path,
    model_name="HCM",
    n_bins=10
)


In [ ]:
generate_and_save_pit_results(
    predictions_HCM_df,
    y_test_array,
    hcm_excel_path,
    model_name="HCM",
    n_bins=10
)


In [ ]:
generate_and_save_calibration_curve(
    predictions_HCM_df,
    y_test_array,
    hcm_excel_path,
    model_name="HCM",
    n_quantiles=10
)


In [ ]:
generate_and_save_error_alignment(predictions_HCM_df, hcm_excel_path, model_name='HCM')
with pd.ExcelWriter(hcm_excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    training_history_df.to_excel(writer, sheet_name='Training_History', index=False)


# **Matrix Evaulation**


In [ ]:
matrix_evaluation_df = build_matrix_evaluation(predictions_HCM_df, model_name='HCM')
matrix_excel_path = resolved_output_folder / 'Matrix Evaluation.xlsx'
with pd.ExcelWriter(matrix_excel_path, engine='xlsxwriter') as writer:
    plot_metrics_and_save_to_excel(writer, predictions_HCM_df, y_test_array.reshape(-1), 'HCM')
    matrix_evaluation_df.to_excel(writer, sheet_name='Matrix Evaluation', index=False)

print(matrix_evaluation_df)
print(f"Saved matrix evaluation to: {matrix_excel_path}")
